## 1. Setup & Konfigurasi

In [ ]:
import os
import re
import csv
import glob
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore', category=UserWarning)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Tidak berjalan di Colab — lewati mount Google Drive.')

Mounted at /content/drive


In [ ]:
BASE_DIR = '/content/drive/MyDrive/DATA_SKRIPSI'
RAW_DIR = os.path.join(BASE_DIR, 'data_mentah')
OUT_DIR = os.path.join(BASE_DIR, 'data_processed')

TAHUN_AWAL, TAHUN_AKHIR = 2014, 2024
TAHUN_UJI = [2023, 2024]          # data latih 2014–2022, data uji 2023–2024

# folder tiap variabel di RAW_DIR, dan apakah tabelnya dipisah per jenis kelamin
SUMBER = {
    'rls': dict(folder='rls', per_jk=False, label='Rata-rata Lama Sekolah (tahun)'),
    'ahh': dict(folder='ahh', per_jk=True,  label='Angka Harapan Hidup (tahun)'),
    'ppk': dict(folder='ppk', per_jk=True,  label='Pengeluaran per Kapita Disesuaikan (ribu Rp/orang/tahun)'),
    'ikk': dict(folder='ikk', per_jk=False, label='Indeks Kemahalan Konstruksi'),
    'p0':  dict(folder='p0',  per_jk=False, label='Persentase Penduduk Miskin (%)'),
}

FITUR = ['rls', 'ahh_l', 'ahh_p', 'ppk_l', 'ppk_p', 'ikk']
TARGET = 'p0'
KOLOM_ID = ['kode_kabkota', 'nama_kabkota', 'provinsi', 'jenis', 'tahun']

# Rentang wajar dipakai untuk mendeteksi SALAH BACA angka (mis. koma/titik tertukar)
RENTANG_WAJAR = {
    'rls':   (0, 16),
    'ahh_l': (50, 90),
    'ahh_p': (50, 90),
    'ppk_l': (1_000, 60_000),
    'ppk_p': (1_000, 60_000),
    'ikk':   (50, 200),
    'p0':    (0, 100),
}

os.makedirs(OUT_DIR, exist_ok=True)
print('RAW_DIR :', RAW_DIR)
print('OUT_DIR :', OUT_DIR)
for var, cfg in SUMBER.items():
    files = sorted(glob.glob(os.path.join(RAW_DIR, cfg['folder'], '*')))
    print(f"  {var:4s}: {len(files)} file -> {[os.path.basename(f) for f in files]}")

RAW_DIR : /content/drive/MyDrive/DATA_SKRIPSI/data_mentah
OUT_DIR : /content/drive/MyDrive/DATA_SKRIPSI/data_processed
  rls : 11 file -> ['[Metode Baru] Rata-rata Lama Sekolah, 2014.xlsx', '[Metode Baru] Rata-rata Lama Sekolah, 2015.xlsx', '[Metode Baru] Rata-rata Lama Sekolah, 2016.xlsx', '[Metode Baru] Rata-rata Lama Sekolah, 2017.xlsx', '[Metode Baru] Rata-rata Lama Sekolah, 2018.xlsx', '[Metode Baru] Rata-rata Lama Sekolah, 2019.xlsx', '[Metode Baru] Rata-rata Lama Sekolah, 2020.xlsx', '[Metode Baru] Rata-rata Lama Sekolah, 2021.xlsx', '[Metode Baru] Rata-rata Lama Sekolah, 2022.xlsx', '[Metode Baru] Rata-rata Lama Sekolah, 2023.xlsx', '[Metode Baru] Rata-rata Lama Sekolah, 2024.xlsx']
  ahh : 11 file -> ['Angka Harapan Hidup (AHH) Menurut Kabupaten_Kota dan Jenis Kelamin, 2014.xlsx', 'Angka Harapan Hidup (AHH) Menurut Kabupaten_Kota dan Jenis Kelamin, 2015.xlsx', 'Angka Harapan Hidup (AHH) Menurut Kabupaten_Kota dan Jenis Kelamin, 2016.xlsx', 'Angka Harapan Hidup (AHH) Menurut Ka

## 2. Daftar 119 Kabupaten/Kota Pulau Jawa

In [ ]:
PROVINSI = {
    '31': 'DKI Jakarta', '32': 'Jawa Barat', '33': 'Jawa Tengah',
    '34': 'DI Yogyakarta', '35': 'Jawa Timur', '36': 'Banten',
}

MASTER_JAWA = [
    # DKI Jakarta
    ('3101', 'Kepulauan Seribu', 'Kabupaten'),
    ('3171', 'Jakarta Selatan', 'Kota'), ('3172', 'Jakarta Timur', 'Kota'),
    ('3173', 'Jakarta Pusat', 'Kota'), ('3174', 'Jakarta Barat', 'Kota'),
    ('3175', 'Jakarta Utara', 'Kota'),
    # Jawa Barat
    ('3201', 'Bogor', 'Kabupaten'), ('3202', 'Sukabumi', 'Kabupaten'),
    ('3203', 'Cianjur', 'Kabupaten'), ('3204', 'Bandung', 'Kabupaten'),
    ('3205', 'Garut', 'Kabupaten'), ('3206', 'Tasikmalaya', 'Kabupaten'),
    ('3207', 'Ciamis', 'Kabupaten'), ('3208', 'Kuningan', 'Kabupaten'),
    ('3209', 'Cirebon', 'Kabupaten'), ('3210', 'Majalengka', 'Kabupaten'),
    ('3211', 'Sumedang', 'Kabupaten'), ('3212', 'Indramayu', 'Kabupaten'),
    ('3213', 'Subang', 'Kabupaten'), ('3214', 'Purwakarta', 'Kabupaten'),
    ('3215', 'Karawang', 'Kabupaten'), ('3216', 'Bekasi', 'Kabupaten'),
    ('3217', 'Bandung Barat', 'Kabupaten'), ('3218', 'Pangandaran', 'Kabupaten'),
    ('3271', 'Bogor', 'Kota'), ('3272', 'Sukabumi', 'Kota'),
    ('3273', 'Bandung', 'Kota'), ('3274', 'Cirebon', 'Kota'),
    ('3275', 'Bekasi', 'Kota'), ('3276', 'Depok', 'Kota'),
    ('3277', 'Cimahi', 'Kota'), ('3278', 'Tasikmalaya', 'Kota'),
    ('3279', 'Banjar', 'Kota'),
    # Jawa Tengah
    ('3301', 'Cilacap', 'Kabupaten'), ('3302', 'Banyumas', 'Kabupaten'),
    ('3303', 'Purbalingga', 'Kabupaten'), ('3304', 'Banjarnegara', 'Kabupaten'),
    ('3305', 'Kebumen', 'Kabupaten'), ('3306', 'Purworejo', 'Kabupaten'),
    ('3307', 'Wonosobo', 'Kabupaten'), ('3308', 'Magelang', 'Kabupaten'),
    ('3309', 'Boyolali', 'Kabupaten'), ('3310', 'Klaten', 'Kabupaten'),
    ('3311', 'Sukoharjo', 'Kabupaten'), ('3312', 'Wonogiri', 'Kabupaten'),
    ('3313', 'Karanganyar', 'Kabupaten'), ('3314', 'Sragen', 'Kabupaten'),
    ('3315', 'Grobogan', 'Kabupaten'), ('3316', 'Blora', 'Kabupaten'),
    ('3317', 'Rembang', 'Kabupaten'), ('3318', 'Pati', 'Kabupaten'),
    ('3319', 'Kudus', 'Kabupaten'), ('3320', 'Jepara', 'Kabupaten'),
    ('3321', 'Demak', 'Kabupaten'), ('3322', 'Semarang', 'Kabupaten'),
    ('3323', 'Temanggung', 'Kabupaten'), ('3324', 'Kendal', 'Kabupaten'),
    ('3325', 'Batang', 'Kabupaten'), ('3326', 'Pekalongan', 'Kabupaten'),
    ('3327', 'Pemalang', 'Kabupaten'), ('3328', 'Tegal', 'Kabupaten'),
    ('3329', 'Brebes', 'Kabupaten'),
    ('3371', 'Magelang', 'Kota'), ('3372', 'Surakarta', 'Kota'),
    ('3373', 'Salatiga', 'Kota'), ('3374', 'Semarang', 'Kota'),
    ('3375', 'Pekalongan', 'Kota'), ('3376', 'Tegal', 'Kota'),
    # DI Yogyakarta
    ('3401', 'Kulon Progo', 'Kabupaten'), ('3402', 'Bantul', 'Kabupaten'),
    ('3403', 'Gunungkidul', 'Kabupaten'), ('3404', 'Sleman', 'Kabupaten'),
    ('3471', 'Yogyakarta', 'Kota'),
    # Jawa Timur
    ('3501', 'Pacitan', 'Kabupaten'), ('3502', 'Ponorogo', 'Kabupaten'),
    ('3503', 'Trenggalek', 'Kabupaten'), ('3504', 'Tulungagung', 'Kabupaten'),
    ('3505', 'Blitar', 'Kabupaten'), ('3506', 'Kediri', 'Kabupaten'),
    ('3507', 'Malang', 'Kabupaten'), ('3508', 'Lumajang', 'Kabupaten'),
    ('3509', 'Jember', 'Kabupaten'), ('3510', 'Banyuwangi', 'Kabupaten'),
    ('3511', 'Bondowoso', 'Kabupaten'), ('3512', 'Situbondo', 'Kabupaten'),
    ('3513', 'Probolinggo', 'Kabupaten'), ('3514', 'Pasuruan', 'Kabupaten'),
    ('3515', 'Sidoarjo', 'Kabupaten'), ('3516', 'Mojokerto', 'Kabupaten'),
    ('3517', 'Jombang', 'Kabupaten'), ('3518', 'Nganjuk', 'Kabupaten'),
    ('3519', 'Madiun', 'Kabupaten'), ('3520', 'Magetan', 'Kabupaten'),
    ('3521', 'Ngawi', 'Kabupaten'), ('3522', 'Bojonegoro', 'Kabupaten'),
    ('3523', 'Tuban', 'Kabupaten'), ('3524', 'Lamongan', 'Kabupaten'),
    ('3525', 'Gresik', 'Kabupaten'), ('3526', 'Bangkalan', 'Kabupaten'),
    ('3527', 'Sampang', 'Kabupaten'), ('3528', 'Pamekasan', 'Kabupaten'),
    ('3529', 'Sumenep', 'Kabupaten'),
    ('3571', 'Kediri', 'Kota'), ('3572', 'Blitar', 'Kota'),
    ('3573', 'Malang', 'Kota'), ('3574', 'Probolinggo', 'Kota'),
    ('3575', 'Pasuruan', 'Kota'), ('3576', 'Mojokerto', 'Kota'),
    ('3577', 'Madiun', 'Kota'), ('3578', 'Surabaya', 'Kota'),
    ('3579', 'Batu', 'Kota'),
    # Banten
    ('3601', 'Pandeglang', 'Kabupaten'), ('3602', 'Lebak', 'Kabupaten'),
    ('3603', 'Tangerang', 'Kabupaten'), ('3604', 'Serang', 'Kabupaten'),
    ('3671', 'Tangerang', 'Kota'), ('3672', 'Cilegon', 'Kota'),
    ('3673', 'Serang', 'Kota'), ('3674', 'Tangerang Selatan', 'Kota'),
]

master = pd.DataFrame(MASTER_JAWA, columns=['kode_kabkota', 'nama', 'jenis'])
master['provinsi'] = master['kode_kabkota'].str[:2].map(PROVINSI)
master['nama_kabkota'] = np.where(master['jenis'] == 'Kota',
                                  'Kota ' + master['nama'], 'Kab. ' + master['nama'])
master = master[['kode_kabkota', 'nama_kabkota', 'provinsi', 'jenis', 'nama']]

assert len(master) == 119 and master['kode_kabkota'].is_unique
print(master.groupby(['provinsi', 'jenis']).size().unstack(fill_value=0))
print('Total:', len(master))

jenis          Kabupaten  Kota
provinsi                      
Banten                 4     4
DI Yogyakarta          4     1
DKI Jakarta            1     5
Jawa Barat            18     9
Jawa Tengah           29     6
Jawa Timur            29     9
Total: 119


## 3. Normalisasi Nama & Konversi Angka

In [ ]:
def normalisasi_nama(teks):
    """Ubah nama wilayah menjadi (jenis, inti_nama) yang seragam.
    'Kab. Adm. Kep. Seribu' -> ('Kabupaten', 'KEPULAUANSERIBU')
    'Kota Adm. Jakarta Selatan' -> ('Kota', 'JAKARTASELATAN')
    'Gunung Kidul' -> (None, 'GUNUNGKIDUL')   # tanpa awalan
    """
    t = str(teks).upper()
    t = re.sub(r'\(.*?\)', ' ', t)                 # buang keterangan dalam kurung
    t = re.sub(r'[^A-Z]', ' ', t)                  # buang titik, strip, angka, dll.
    t = re.sub(r'\b(ADM|ADMINISTRASI)\b', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()

    jenis = None
    m = re.match(r'^(KABUPATEN|KAB)\s+(.+)$', t)
    if m:
        jenis, t = 'Kabupaten', m.group(2)
    else:
        m = re.match(r'^KOTA\s+(.+)$', t)
        if m:
            jenis, t = 'Kota', m.group(1)
    t = re.sub(r'\bKEP\b', 'KEPULAUAN', t)
    return jenis, t.replace(' ', '')


_IDX_JENIS = {}
for _, r in master.iterrows():
    _, inti = normalisasi_nama(r['nama'])
    _IDX_JENIS[(r['jenis'], inti)] = r['kode_kabkota']

KOTA_TANPA_AWALAN = {'JAKARTASELATAN', 'JAKARTATIMUR', 'JAKARTAPUSAT',
                     'JAKARTABARAT', 'JAKARTAUTARA', 'YOGYAKARTA'}


def cocokkan_kode(teks):
    """Kembalikan kode BPS kab/kota Jawa, atau None jika bukan kab/kota Jawa
    (baris provinsi, wilayah luar Jawa, catatan kaki, dsb.)."""
    if pd.isna(teks) or str(teks).strip() == '':
        return None
    jenis, inti = normalisasi_nama(teks)
    if jenis is not None:
        return _IDX_JENIS.get((jenis, inti))
    kode = _IDX_JENIS.get(('Kabupaten', inti))
    if kode is None and inti in KOTA_TANPA_AWALAN:
        kode = _IDX_JENIS.get(('Kota', inti))
    return kode


PROV_JAWA = {'DKIJAKARTA', 'DAERAHKHUSUSIBUKOTAJAKARTA', 'DAERAHKHUSUSJAKARTA', 'JAWABARAT',
             'JAWATENGAH', 'DIYOGYAKARTA', 'DAERAHISTIMEWAYOGYAKARTA', 'JAWATIMUR', 'BANTEN',
             'WESTJAVA', 'CENTRALJAVA', 'EASTJAVA'}
PROV_LUAR_JAWA = {
    'ACEH', 'SUMATERAUTARA', 'SUMATERABARAT', 'RIAU', 'JAMBI', 'SUMATERASELATAN', 'BENGKULU',
    'LAMPUNG', 'KEPULAUANBANGKABELITUNG', 'BANGKABELITUNG', 'KEPULAUANRIAU', 'BALI',
    'NUSATENGGARABARAT', 'NUSATENGGARATIMUR', 'KALIMANTANBARAT', 'KALIMANTANTENGAH',
    'KALIMANTANSELATAN', 'KALIMANTANTIMUR', 'KALIMANTANUTARA', 'SULAWESIUTARA', 'SULAWESITENGAH',
    'SULAWESISELATAN', 'SULAWESITENGGARA', 'GORONTALO', 'SULAWESIBARAT', 'MALUKU', 'MALUKUUTARA',
    'PAPUABARAT', 'PAPUABARATDAYA', 'PAPUA', 'PAPUASELATAN', 'PAPUATENGAH', 'PAPUAPEGUNUNGAN',
    'NORTHSUMATERA', 'WESTSUMATERA', 'SOUTHSUMATERA', 'RIAUISLANDS', 'WESTNUSATENGGARA',
    'EASTNUSATENGGARA', 'WESTKALIMANTAN', 'CENTRALKALIMANTAN', 'SOUTHKALIMANTAN',
    'EASTKALIMANTAN', 'NORTHKALIMANTAN', 'NORTHSULAWESI', 'CENTRALSULAWESI', 'SOUTHSULAWESI',
    'SOUTHEASTSULAWESI', 'WESTSULAWESI', 'NORTHMALUKU', 'WESTPAPUA', 'SOUTHWESTPAPUA',
    'SOUTHPAPUA', 'CENTRALPAPUA', 'HIGHLANDPAPUA', 'INDONESIA'}


def kelompok_provinsi(teks):
    """'jawa' / 'luar' bila teks adalah baris judul provinsi, selain itu None."""
    if pd.isna(teks) or str(teks).strip() == '':
        return None
    jenis, inti = normalisasi_nama(teks)
    if jenis is not None:
        return None
    inti = re.sub(r'^PROVINSI', '', inti)
    if inti in PROV_JAWA:
        return 'jawa'
    if inti in PROV_LUAR_JAWA:
        return 'luar'
    return None


def ke_angka(x, boleh_pemisah_ribuan=False):
    """Konversi sel menjadi float. Menangani '-', '…', format Indonesia (8,53) dan
    pemisah ribuan (10.345) untuk variabel bernilai ribuan seperti pengeluaran per kapita."""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)
    s = str(x).strip().replace(' ', '').replace(' ', '')
    if s in ('', '-', '–', '—', '…', '...', 'NA', 'N/A', 'nan', 'None', '*'):
        return np.nan
    s = re.sub(r'[^0-9,.\-]', '', s)
    if ',' in s and '.' in s:
        if s.rfind(',') > s.rfind('.'):          # 10.345,67 (format Indonesia)
            s = s.replace('.', '').replace(',', '.')
        else:                                     # 10,345.67 (format Inggris)
            s = s.replace(',', '')
    elif ',' in s:
        if boleh_pemisah_ribuan and re.fullmatch(r'-?\d{1,3}(,\d{3})+', s):
            s = s.replace(',', '')
        else:
            s = s.replace(',', '.')
    elif '.' in s and boleh_pemisah_ribuan and re.fullmatch(r'-?\d{1,3}(\.\d{3})+', s):
        s = s.replace('.', '')
    try:
        return float(s)
    except ValueError:
        return np.nan


# uji cepat
for contoh in ['Kab. Adm. Kep. Seribu', 'Kota Adm. Jakarta Selatan', 'Gunung Kidul',
               'KULONPROGO', 'Kota Bogor', 'Bogor', 'Yogyakarta', 'Kota Banjar',
               'Banjar', 'JAWA BARAT', 'Kota Surakarta (Solo)']:
    print(f'{contoh:28s} -> {cocokkan_kode(contoh)}')
print(ke_angka('8,53'), ke_angka('10.345', True), ke_angka('69.875'), ke_angka('-'))

Kab. Adm. Kep. Seribu        -> 3101
Kota Adm. Jakarta Selatan    -> 3171
Gunung Kidul                 -> 3403
KULONPROGO                   -> 3401
Kota Bogor                   -> 3271
Bogor                        -> 3201
Yogyakarta                   -> 3471
Kota Banjar                  -> 3279
Banjar                       -> None
JAWA BARAT                   -> None
Kota Surakarta (Solo)        -> 3372
8.53 10345.0 69.875 nan


## 4. Pembaca Tabel BPS

In [ ]:
POLA_TAHUN_PENUH = re.compile(r'^(20\d{2})(\.0+)?$')
POLA_TAHUN_DALAM = re.compile(r'(?<!\d)(20\d{2})(?!\d)')
PENANDA_LAIN = '__lain__'


def _baca_csv_mentah(path):
    for enc in ('utf-8-sig', 'cp1252', 'latin-1'):
        try:
            with open(path, encoding=enc) as f:
                teks = f.read()
            break
        except UnicodeDecodeError:
            continue
    baris = teks.splitlines()
    contoh = '\n'.join(baris[:50])
    delim = max([',', ';', '\t', '|'], key=contoh.count)
    rows = list(csv.reader(baris, delimiter=delim))
    lebar = max((len(r) for r in rows), default=0)
    rows = [r + [None] * (lebar - len(r)) for r in rows]
    df = pd.DataFrame(rows).replace({'': None})
    return {'csv': df}


def baca_file_mentah(path):
    ext = os.path.splitext(path)[1].lower()
    if ext in ('.xlsx', '.xlsm', '.xls'):
        # tanpa dtype=str agar angka Excel tetap angka
        try:
            return pd.read_excel(path, sheet_name=None, header=None)
        except ImportError:
            raise ImportError('File .xls butuh library xlrd: jalankan !pip install xlrd')
    if ext in ('.csv', '.txt'):
        return _baca_csv_mentah(path)
    raise ValueError(f'Format tidak didukung: {path}')


def _tahun_sel(v, boleh_di_dalam):
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    m = POLA_TAHUN_PENUH.match(s)
    if m:
        return int(m.group(1))
    if boleh_di_dalam:
        m = POLA_TAHUN_DALAM.search(s)
        if m:
            return int(m.group(1))
    return PENANDA_LAIN


def _jk_sel(v):
    if pd.isna(v):
        return np.nan
    t = str(v).lower()
    laki = ('laki' in t) or bool(re.search(r'\bmale\b', t))
    pr = ('perempuan' in t) or ('wanita' in t) or ('female' in t)
    if laki and not pr:
        return 'l'
    if pr and not laki:
        return 'p'
    return PENANDA_LAIN                              # mis. "Laki-laki + Perempuan", "Total"


def _isi_ke_kanan(nilai):
    """Forward-fill horizontal untuk sel header yang di-merge."""
    hasil, terakhir = [], None
    for v in nilai:
        kosong = v is None or (isinstance(v, float) and np.isnan(v))
        if not kosong:
            terakhir = v
        hasil.append(None if terakhir == PENANDA_LAIN else terakhir)
    return hasil


def parse_tabel(df, per_jk, tahun_file=None, var='', sumber=''):
    df = df.dropna(how='all').dropna(axis=1, how='all').reset_index(drop=True)
    if df.empty:
        return pd.DataFrame()
    df.columns = range(df.shape[1])

    # 1) kolom nama wilayah
    skor = {c: df[c].map(cocokkan_kode).notna().sum() for c in df.columns}
    kol_nama = max(skor, key=skor.get)
    if skor[kol_nama] == 0:
        return pd.DataFrame()
    kode = df[kol_nama].map(cocokkan_kode)

    konteks, sekarang = [], None
    for nm in df[kol_nama]:
        g = kelompok_provinsi(nm)
        sekarang = g if g is not None else sekarang
        konteks.append(sekarang)
    kode = kode.where(pd.Series(konteks, index=kode.index) != 'luar')
    if kode.notna().sum() == 0:
        return pd.DataFrame()
    baris_data_pertama = kode.first_valid_index()
    n_head = baris_data_pertama                    # baris di atas data = area header

    # 2) baris header tahun
    label_tahun = None
    for boleh_di_dalam in (False, True):
        for i in range(min(n_head, 25)):              # header ada di baris-baris teratas
            vals = [_tahun_sel(v, boleh_di_dalam) for v in df.iloc[i]]
            jml = sum(isinstance(v, int) for v in vals)
            if jml >= (1 if not boleh_di_dalam else 2):
                label_tahun = _isi_ke_kanan(vals)
                break
        if label_tahun is not None:
            break

    # 3) baris header jenis kelamin
    label_jk = None
    if per_jk:
        for i in range(min(n_head, 25)):
            vals = [_jk_sel(v) for v in df.iloc[i]]
            if any(v in ('l', 'p') for v in vals):
                label_jk = _isi_ke_kanan(vals)
                break
        if label_jk is None:
            raise ValueError(f'[{var}] {sumber}: header Laki-laki/Perempuan tidak ditemukan.')

    if label_tahun is None:
        if tahun_file is None:
            raise ValueError(f'[{var}] {sumber}: header tahun tidak ditemukan dan nama file '
                             f'tidak mengandung tahun. Tambahkan tahun pada nama file, mis. {var}_2019.xlsx')
        label_tahun = [tahun_file] * df.shape[1]

    data = df.loc[kode.notna()]
    kode = kode.loc[kode.notna()]

    # 4) kolom nilai: punya label tahun (dan label jk bila per_jk), bukan kolom nama/kode
    kolom_nilai = []
    for c in df.columns:
        if c == kol_nama or label_tahun[c] is None:
            continue
        if per_jk and label_jk[c] is None:
            continue
        isi = data[c].dropna().astype(str)
        if len(isi) and isi.str.fullmatch(r'\d{4}(\.0)?').all() and \
                isi.str[:2].isin(PROVINSI.keys()).all():
            continue                                  # kolom kode wilayah BPS (mis. 3201)
        kolom_nilai.append(c)

    if not kolom_nilai:
        raise ValueError(f'[{var}] {sumber}: kolom nilai tidak ditemukan.')
    if not per_jk:
        thn = [label_tahun[c] for c in kolom_nilai]
        if len(thn) != len(set(thn)):
            raise ValueError(f'[{var}] {sumber}: lebih dari satu kolom untuk tahun yang sama {thn}. '
                             'Periksa apakah tabel berisi lebih dari satu indikator.')

    ribuan = (var == 'ppk')
    hasil = []
    for c in kolom_nilai:
        hasil.append(pd.DataFrame({
            'kode_kabkota': kode.values,
            'nama_asli': data[kol_nama].values,
            'tahun': int(label_tahun[c]),
            'jk': label_jk[c] if per_jk else '-',
            'nilai': [ke_angka(v, ribuan) for v in data[c].values],
            'sumber': sumber,
        }))
    return pd.concat(hasil, ignore_index=True)


def baca_variabel(var, cfg):
    folder = os.path.join(RAW_DIR, cfg['folder'])
    files = sorted(f for f in glob.glob(os.path.join(folder, '*'))
                   if os.path.splitext(f)[1].lower() in ('.xlsx', '.xlsm', '.xls', '.csv', '.txt')
                   and not os.path.basename(f).startswith('~$'))
    if not files:
        raise FileNotFoundError(f'Tidak ada file di {folder}')
    potongan = []
    for f in files:
        m = POLA_TAHUN_DALAM.search(os.path.basename(f))
        tahun_file = int(m.group(1)) if m else None
        for sheet, df in baca_file_mentah(f).items():
            nama_sumber = os.path.basename(f) + ('' if sheet == 'csv' else f' [{sheet}]')
            hasil = parse_tabel(df, cfg['per_jk'], tahun_file, var, nama_sumber)
            if not hasil.empty:
                potongan.append(hasil)
    if not potongan:
        raise ValueError(f'[{var}] tidak ada baris kab/kota Jawa yang terbaca dari {folder}')
    long = pd.concat(potongan, ignore_index=True)
    long = long[long['tahun'].between(TAHUN_AWAL, TAHUN_AKHIR)]

    # duplikat (wilayah, tahun, jk): satu kab/kota-tahun muncul lebih dari sekali
    kunci = ['kode_kabkota', 'tahun', 'jk']
    dup = long[long.duplicated(kunci, keep=False)]
    if not dup.empty:
        konflik = dup[dup.groupby(kunci)['nilai'].transform('nunique') > 1]
        if len(konflik):
            n = konflik[kunci].drop_duplicates().shape[0]
            print(f'  ⚠ [{var}] {n} kombinasi wilayah-tahun punya nilai BERBEDA. Lihat kolom nama_asli: '
                  'jika namanya berbeda, ada dua wilayah yang terbaca sebagai kab/kota yang sama. '
                  'Sementara dipakai nilai dari file terakhir (urut nama).')
            print(konflik.sort_values(kunci)[['kode_kabkota', 'tahun', 'jk', 'nama_asli', 'nilai', 'sumber']]
                  .head(12).to_string(index=False))
        long = long.drop_duplicates(kunci, keep='last')

    long['variabel'] = var if not cfg['per_jk'] else var + '_' + long['jk']
    return long.reset_index(drop=True)

## 5. Membaca Semua Sumber + Laporan Kecocokan Wilayah

In [ ]:
semua_long = []
for var, cfg in SUMBER.items():
    print(f'\n=== {var.upper()} — {cfg["label"]} ===')
    long = baca_variabel(var, cfg)
    semua_long.append(long)

    for v, g in long.groupby('variabel'):
        per_tahun = g.dropna(subset=['nilai']).groupby('tahun')['kode_kabkota'].nunique()
        per_tahun = per_tahun.reindex(range(TAHUN_AWAL, TAHUN_AKHIR + 1), fill_value=0)
        print(f'  {v}: kab/kota terisi per tahun ->', {int(k): int(n) for k, n in per_tahun.items()})
        hilang = set(master['kode_kabkota']) - set(g['kode_kabkota'])
        if hilang:
            nama = master.set_index('kode_kabkota').loc[sorted(hilang), 'nama_kabkota'].tolist()
            print(f'  ⚠ {len(hilang)} kab/kota tidak ditemukan sama sekali: {nama}')

data_long = pd.concat(semua_long, ignore_index=True)
print('\nTotal baris format panjang:', len(data_long))


=== RLS — Rata-rata Lama Sekolah (tahun) ===
  rls: kab/kota terisi per tahun -> {2014: 119, 2015: 119, 2016: 119, 2017: 119, 2018: 119, 2019: 119, 2020: 119, 2021: 119, 2022: 119, 2023: 119, 2024: 119}

=== AHH — Angka Harapan Hidup (tahun) ===
  ahh_l: kab/kota terisi per tahun -> {2014: 119, 2015: 119, 2016: 0, 2017: 119, 2018: 119, 2019: 119, 2020: 119, 2021: 119, 2022: 119, 2023: 119, 2024: 119}
  ahh_p: kab/kota terisi per tahun -> {2014: 119, 2015: 119, 2016: 0, 2017: 119, 2018: 119, 2019: 119, 2020: 119, 2021: 119, 2022: 119, 2023: 119, 2024: 119}

=== PPK — Pengeluaran per Kapita Disesuaikan (ribu Rp/orang/tahun) ===
  ppk_l: kab/kota terisi per tahun -> {2014: 119, 2015: 119, 2016: 0, 2017: 119, 2018: 119, 2019: 119, 2020: 119, 2021: 119, 2022: 119, 2023: 119, 2024: 119}
  ppk_p: kab/kota terisi per tahun -> {2014: 119, 2015: 119, 2016: 0, 2017: 119, 2018: 119, 2019: 119, 2020: 119, 2021: 119, 2022: 119, 2023: 119, 2024: 119}

=== IKK — Indeks Kemahalan Konstruksi ===
  ikk:

## 6. Merge Menjadi Satu Tabel Panel

In [ ]:
lebar = data_long.pivot_table(index=['kode_kabkota', 'tahun'], columns='variabel',
                              values='nilai', aggfunc='first').reset_index()
lebar.columns.name = None

kerangka = master[['kode_kabkota', 'nama_kabkota', 'provinsi', 'jenis']].merge(
    pd.DataFrame({'tahun': range(TAHUN_AWAL, TAHUN_AKHIR + 1)}), how='cross')

panel = kerangka.merge(lebar, on=['kode_kabkota', 'tahun'], how='left')
for kol in FITUR + [TARGET]:
    if kol not in panel.columns:
        raise KeyError(f'Kolom {kol} tidak terbentuk. Periksa sumber datanya.')
panel = panel[KOLOM_ID + FITUR + [TARGET]].sort_values(['kode_kabkota', 'tahun']).reset_index(drop=True)

print('Ukuran panel:', panel.shape, '(seharusnya 119 × 11 = 1.309 baris)')
panel.head(12)

Ukuran panel: (1309, 12) (seharusnya 119 × 11 = 1.309 baris)


,kode_kabkota,nama_kabkota,provinsi,jenis,tahun,rls,ahh_l,ahh_p,ppk_l,ppk_p,ikk,p0
0,3101,Kab. Kepulauan Seribu,DKI Jakarta,Kabupaten,2014,8.03,65.51,68.83,16054.10,10887.55,NaN,11.56
1,3101,Kab. Kepulauan Seribu,DKI Jakarta,Kabupaten,2015,8.04,66.01,69.33,16107.58,11077.58,NaN,11.40
2,3101,Kab. Kepulauan Seribu,DKI Jakarta,Kabupaten,2016,8.24,NaN,NaN,NaN,NaN,133.29,12.58
3,3101,Kab. Kepulauan Seribu,DKI Jakarta,Kabupaten,2017,8.25,66.10,69.96,16680.00,11406.00,127.10,12.98
4,3101,Kab. Kepulauan Seribu,DKI Jakarta,Kabupaten,2018,8.46,66.30,70.22,17196.00,11765.00,116.74,11.98
5,3101,Kab. Kepulauan Seribu,DKI Jakarta,Kabupaten,2019,8.47,66.47,70.46,17331.00,12130.00,125.15,12.09
6,3101,Kab. Kepulauan Seribu,DKI Jakarta,Kabupaten,2020,8.68,66.67,70.74,17054.00,11925.00,128.37,14.87
7,3101,Kab. Kepulauan Seribu,DKI Jakarta,Kabupaten,2021,8.81,66.84,71.03,17227.00,12201.00,131.85,15.06
8,3101,Kab. Kepulauan Seribu,DKI Jakarta,Kabupaten,2022,9.02,67.02,71.27,17804.00,12587.00,126.62,14.11
9,3101,Kab. Kepulauan Seribu,DKI Jakarta,Kabupaten,2023,9.03,67.68,71.63,18385.00,13041.00,123.98,13.13


## 7. Validasi Data

In [ ]:
# 7a. duplikat kunci
n_dup = panel.duplicated(['kode_kabkota', 'tahun']).sum()
print('Duplikat kode_kabkota+tahun :', n_dup)

# 7b. nilai di luar rentang wajar (indikasi salah baca angka)
print('\nNilai di luar rentang wajar:')
ada_aneh = False
for kol, (lo, hi) in RENTANG_WAJAR.items():
    aneh = panel[(panel[kol] < lo) | (panel[kol] > hi)]
    if len(aneh):
        ada_aneh = True
        print(f'  ⚠ {kol}: {len(aneh)} baris di luar [{lo}, {hi}]')
        print(aneh[['nama_kabkota', 'tahun', kol]].head(10).to_string(index=False))
if not ada_aneh:
    print('  ✓ semua nilai dalam rentang wajar')

# 7c. pemeriksaan logika: AHH perempuan umumnya > AHH laki-laki
n_ahh = (panel['ahh_p'] < panel['ahh_l']).sum()
print(f'\nBaris dengan AHH perempuan < laki-laki: {n_ahh} (umumnya 0; jika banyak, kolom L/P mungkin tertukar)')

# 7d. kelengkapan
print('\nJumlah missing per kolom:')
print(panel[FITUR + [TARGET]].isna().sum().to_string())

Duplikat kode_kabkota+tahun : 0

Nilai di luar rentang wajar:
  ✓ semua nilai dalam rentang wajar

Baris dengan AHH perempuan < laki-laki: 0 (umumnya 0; jika banyak, kolom L/P mungkin tertukar)

Jumlah missing per kolom:
rls        0
ahh_l    119
ahh_p    119
ppk_l    119
ppk_p    119
ikk       12
p0         1


## 8. Penanganan Missing Value

In [ ]:
sebelum = panel.copy()

panel[FITUR] = (panel.groupby('kode_kabkota', group_keys=False)[FITUR]
                .apply(lambda g: g.interpolate(method='linear', limit_area='inside')))

log_imputasi = []
for kol in FITUR:
    m = sebelum[kol].isna() & panel[kol].notna()
    for _, r in panel.loc[m].iterrows():
        log_imputasi.append({'kode_kabkota': r['kode_kabkota'], 'nama_kabkota': r['nama_kabkota'],
                             'tahun': r['tahun'], 'kolom': kol, 'nilai_imputasi': round(r[kol], 4)})
log_imputasi = pd.DataFrame(log_imputasi, columns=['kode_kabkota', 'nama_kabkota', 'tahun',
                                                   'kolom', 'nilai_imputasi'])
print('Nilai prediktor yang diinterpolasi:', len(log_imputasi))
if len(log_imputasi):
    print(log_imputasi.head(20).to_string(index=False))

target_kosong = panel[panel[TARGET].isna()]
print(f'\nBaris dengan target kosong (dihapus): {len(target_kosong)}')
if len(target_kosong):
    print(target_kosong[['nama_kabkota', 'tahun']].to_string(index=False))
panel = panel.dropna(subset=[TARGET]).reset_index(drop=True)

sisa = panel[FITUR].isna().sum()
print('\nSisa NaN pada prediktor (ditangani CatBoost):')
print(sisa[sisa > 0].to_string() if sisa.sum() else '  tidak ada')

per_wil = panel.groupby('kode_kabkota')['tahun'].nunique()
print(f'\nPanel seimbang? {"Ya" if (per_wil == TAHUN_AKHIR - TAHUN_AWAL + 1).all() and len(per_wil) == 119 else "Tidak"}'
      f' — {len(per_wil)} kab/kota, {len(panel)} baris')

Nilai prediktor yang diinterpolasi: 476
kode_kabkota          nama_kabkota  tahun kolom  nilai_imputasi
        3101 Kab. Kepulauan Seribu   2016 ahh_l          66.055
        3171  Kota Jakarta Selatan   2016 ahh_l          71.860
        3172    Kota Jakarta Timur   2016 ahh_l          72.170
        3173    Kota Jakarta Pusat   2016 ahh_l          71.795
        3174    Kota Jakarta Barat   2016 ahh_l          71.370
        3175    Kota Jakarta Utara   2016 ahh_l          70.975
        3201            Kab. Bogor   2016 ahh_l          68.645
        3202         Kab. Sukabumi   2016 ahh_l          68.145
        3203          Kab. Cianjur   2016 ahh_l          67.390
        3204          Kab. Bandung   2016 ahh_l          71.105
        3205            Kab. Garut   2016 ahh_l          68.755
        3206      Kab. Tasikmalaya   2016 ahh_l          66.550
        3207           Kab. Ciamis   2016 ahh_l          68.905
        3208         Kab. Kuningan   2016 ahh_l          70.770


## 9. Pemeriksaan Outlier (Metode IQR)

In [ ]:
ringkas_outlier = []
for kol in FITUR + [TARGET]:
    q1, q3 = panel[kol].quantile([0.25, 0.75])
    iqr = q3 - q1
    bawah, atas = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    m = (panel[kol] < bawah) | (panel[kol] > atas)
    wilayah = panel.loc[m, 'nama_kabkota'].value_counts()
    ringkas_outlier.append({'variabel': kol, 'batas_bawah': round(bawah, 2), 'batas_atas': round(atas, 2),
                            'jumlah_outlier': int(m.sum()),
                            'wilayah_terbanyak': ', '.join(wilayah.index[:3])})
pd.DataFrame(ringkas_outlier)

,variabel,batas_bawah,batas_atas,jumlah_outlier,wilayah_terbanyak
0,rls,3.18,13.34,0,
1,ahh_l,64.16,77.42,32,"Kab. Pandeglang, Kab. Serang, Kab. Bondowoso"
2,ahh_p,68.30,80.94,39,"Kab. Pandeglang, Kab. Serang, Kab. Bondowoso"
3,ppk_l,7415.75,23173.75,42,"Kota Jakarta Selatan, Kota Jakarta Utara, Kota..."
4,ppk_p,862.75,18307.03,34,"Kota Jakarta Selatan, Kota Jakarta Barat, Kota..."
5,ikk,82.84,115.31,68,"Kab. Kepulauan Seribu, Kota Jakarta Selatan, K..."
6,p0,-1.42,21.16,18,"Kab. Sampang, Kab. Bangkalan, Kab. Wonosobo"


## 10. Statistik Deskriptif

In [ ]:
print('Seluruh kab/kota:')
print(panel[FITUR + [TARGET]].describe().T.round(2).to_string())
for j in ['Kabupaten', 'Kota']:
    print(f'\n{j} (n wilayah = {panel.loc[panel.jenis == j, "kode_kabkota"].nunique()}):')
    print(panel.loc[panel.jenis == j, FITUR + [TARGET]].describe().T.round(2).to_string())

Seluruh kab/kota:
        count      mean      std      min       25%       50%       75%       max
rls    1308.0      8.24     1.67     3.49      6.99      7.83      9.53     12.12
ahh_l  1308.0     70.64     2.76    61.03     69.14     70.95     72.45     76.40
ahh_p  1308.0     74.49     2.73    64.68     73.04     74.83     76.20     80.38
ppk_l  1308.0  15643.83  3327.89  8787.20  13325.00  15001.28  17264.50  29695.00
ppk_p  1308.0   9919.26  3552.23  3398.30   7404.36   9291.90  11765.43  24545.00
ikk    1296.0     99.52     7.65    74.45     95.02     99.39    103.14    133.29
p0     1308.0     10.11     4.36     1.67      7.05      9.80     12.69     25.80

Kabupaten (n wilayah = 85):
       count      mean      std      min       25%       50%       75%       max
rls    934.0      7.42     1.08     3.49      6.77      7.35      7.98     11.33
ahh_l  934.0     70.33     2.84    61.03     68.73     70.57     72.33     76.37
ahh_p  934.0     74.18     2.81    64.68     72.62    

## 11. Pembagian Data Latih dan Uji Berdasarkan Waktu

In [ ]:
train = panel[~panel['tahun'].isin(TAHUN_UJI)].reset_index(drop=True)
test = panel[panel['tahun'].isin(TAHUN_UJI)].reset_index(drop=True)

print(f'Data latih : {len(train):5d} baris  tahun {train.tahun.min()}–{train.tahun.max()}')
print(f'Data uji   : {len(test):5d} baris  tahun {test.tahun.min()}–{test.tahun.max()}')
print(f'Proporsi   : {len(train)/len(panel):.1%} : {len(test)/len(panel):.1%}')
print('\nPer jenis wilayah:')
print(pd.DataFrame({'latih': train.jenis.value_counts(), 'uji': test.jenis.value_counts()}))

Data latih :  1070 baris  tahun 2014–2022
Data uji   :   238 baris  tahun 2023–2024
Proporsi   : 81.8% : 18.2%

Per jenis wilayah:
           latih  uji
jenis                
Kabupaten    764  170
Kota         306   68


## 12. Menyimpan Hasil
- `panel_bersih.csv` seluruh data panel bersih (dipakai juga oleh dashboard)
- `train.csv`, `test.csv` data latih dan uji
- `rentang_fitur.json` batas nilai fitur dari data latih, untuk validasi input di dashboard
  (model berbasis pohon tidak mengekstrapolasi dengan andal di luar rentang data latih)
- `log_imputasi.csv`, `master_kabkota_jawa.csv`

In [ ]:
panel.to_csv(os.path.join(OUT_DIR, 'panel_bersih.csv'), index=False)
train.to_csv(os.path.join(OUT_DIR, 'train.csv'), index=False)
test.to_csv(os.path.join(OUT_DIR, 'test.csv'), index=False)
log_imputasi.to_csv(os.path.join(OUT_DIR, 'log_imputasi.csv'), index=False)
master.drop(columns='nama').to_csv(os.path.join(OUT_DIR, 'master_kabkota_jawa.csv'), index=False)

rentang = {kol: {'min': float(train[kol].min()), 'max': float(train[kol].max()),
                 'p01': float(train[kol].quantile(0.01)), 'p99': float(train[kol].quantile(0.99)),
                 'label': lbl}
           for kol, lbl in zip(FITUR, ['Rata-rata Lama Sekolah (tahun)',
                                       'Angka Harapan Hidup Laki-laki (tahun)',
                                       'Angka Harapan Hidup Perempuan (tahun)',
                                       'Pengeluaran per Kapita Laki-laki (ribu Rp/tahun)',
                                       'Pengeluaran per Kapita Perempuan (ribu Rp/tahun)',
                                       'Indeks Kemahalan Konstruksi'])}
with open(os.path.join(OUT_DIR, 'rentang_fitur.json'), 'w') as f:
    json.dump({'fitur': FITUR, 'target': TARGET, 'tahun_uji': TAHUN_UJI, 'rentang': rentang}, f, indent=2)

print('Tersimpan di', OUT_DIR)
for f in sorted(os.listdir(OUT_DIR)):
    print(f'  {f:28s} {os.path.getsize(os.path.join(OUT_DIR, f)) / 1024:8.1f} KB')

Tersimpan di /content/drive/MyDrive/DATA_SKRIPSI/data_processed
  log_imputasi.csv                 17.6 KB
  master_kabkota_jawa.csv           4.5 KB
  panel_bersih.csv                114.3 KB
  rentang_fitur.json                1.2 KB
  test.csv                         20.5 KB
  train.csv                        93.9 KB
